# Karar Deneyi: Additive vs Delta Yazım Kuralı (Streaming Mix)

Bu test, HFP'nin yazım (write) kuralına karar vermek için tasarlanmıştır. 
- `additive`: Yeni bilgi doğrudan eski bilgiye eklenir.
- `delta`: Yeni bilgi ile eski bilgi arasındaki fark alınarak güncelleme yapılır.

Mimari olarak en başarılı çürüme (`cubic_flux_chunked`) ve kapasite (`dpfp`) kuralları sabit tutulmuş, 3 farklı seed üzerinden test yapılmaktadır.

In [ ]:
# --- KURULUM ---
import subprocess, sys, os
REPO = "/kaggle/working/HFP"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/kayra-hn/HFP.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.40", "pandas"], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ["HFP_CKPT_DIR"] = "/kaggle/working/ck"
os.environ["PYTHONPATH"] = REPO
os.makedirs("/kaggle/working/ck", exist_ok=True)
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Kurulum tamam!")

In [ ]:
# --- TEST DONGUSU ---
import subprocess

seeds = [0, 1, 2]
write_rules = ["additive", "delta"]
decay = "cubic_flux_chunked"
fmap = "dpfp"
budget = 1800.0  # Saniye (YARI DA KESILMEMESI ICIN 30 DAKIKAYA CIKARILDI)

for w in write_rules:
    for s in seeds:
        print(f"\n{'-'*50}")
        print(f"Calistiriliyor: {w.upper()} | Seed: {s}")
        print(f"{'-'*50}")
        # python review_scripts/streaming_mix.py <write> <fmap> <seed> <decay> <budget>
        cmd = [sys.executable, "review_scripts/streaming_mix.py", w, fmap, str(s), decay, str(budget)]
        subprocess.run(cmd)
        
print("\nTum testler tamamlandi.")

In [ ]:
# --- SONUCLARI TABLOLASTIRMA ---
import pandas as pd
import re
import os

results_file = "/kaggle/working/ck/sm_results.txt"

data = []
if os.path.exists(results_file):
    with open(results_file, "r") as f:
        lines = f.readlines()
    
    for line in lines:
        # Ornek satir: sm_cubic_flux_chunked_additive_dpfp_0 ctx160 single=90.0 upd=45.0 stale=20.0
        match = re.match(r'sm_(.*?)_(.*?)_(.*?)_(\d+) ctx(\d+) single=([\d.]+) upd=([\d.]+) stale=([\d.]+)', line.strip())
        if match:
            decay, write, fmap, seed, ctx, single, upd, stale = match.groups()
            data.append({
                "Write Rule": write.upper(),
                "Seed": int(seed),
                "Ctx": int(ctx),
                "Tek-Yazim (Single) %": float(single),
                "Guncellenen (Upd) %": float(upd),
                "Bayat (Stale) %": float(stale)
            })

    df = pd.DataFrame(data)
    
    if not df.empty:
        # Ortalamalari hesapla
        summary = df.groupby(["Write Rule", "Ctx"]).mean().drop(columns=["Seed"])
        summary = summary.round(1)
        
        print("\n============================================================")
        print("KARAR DENEYI SONUCLARI (3 SEED ORTALAMASI)")
        print("============================================================")
        print(summary)
        print("\n* 'Guncellenen %' ne kadar yuksekse model bilgiyi o kadar iyi guncelleyebilmistir.")
        print("* 'Bayat %' ne kadar dusukse (0'a yakinsa) eski bilgi o kadar iyi silinmistir.\n")
        
        print("TUM DETAYLAR:")
        print(df.sort_values(by=["Write Rule", "Ctx", "Seed"]))
    else:
        print("Tablolanacak sonuc bulunamadi.")
else:
    print(f"Sonuc dosyasi bulunamadi: {results_file}")